## 1. 环境验证：连接你的GPU
在开始一切之前，我们需要确认 PyTorch 已经正确识别到了你的 NVIDIA 显卡。这是所有深度学习任务的物理前提。

In [2]:
import torch

# 验证 PyTorch 安装与 GPU 状态
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 是否可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"当前显卡型号: {torch.cuda.get_device_name(0)}")
    print(f"当前显存总量: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("未检测到可用 GPU，请检查驱动或安装版本！")

c:\software\Anaconda3\envs\llm-finetune\lib\site-packages\torch\_subclasses\functional_tensor.py:276: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


PyTorch 版本: 2.7.0+cu128
CUDA 是否可用: True
当前显卡型号: NVIDIA GeForce GTX 1650 Ti
当前显存总量: 4.00 GB


## 张量 (Tensor) 的物理感知
### 1. 常用张量创建函数
在实际开发中，我们很少手动输入每个数字，而是使用内置函数来生成特定用途的张量

注意：
**张量的维度是从外向内看的**

In [5]:
import torch

# 1.1 随机初始化：常用于模拟权重或输入
# 创建一个 2x3 的矩阵，数值服从标准正态分布
rand_tensor = torch.randn(2, 3) 

# 1.2 全 0 或全 1 初始化：常用于创建掩码（Mask）
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)

# 1.3 序列初始化：类似于 Python 的 range()
# 常用于生成位置索引（Position IDs）
indices = torch.arange(0, 10) 

print(f"随机张量:\n{rand_tensor}")
print(f"全零张量:", zeros)
print(f"索引张量: {indices}")

随机张量:
tensor([[ 0.7843, -1.2032, -0.0467],
        [ 1.0964,  0.8680,  2.1301]])
全零张量: tensor([[0., 0., 0.],
        [0., 0., 0.]])
索引张量: tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


### 2. 张量的三大基本属性
这三个属性决定了一个张量能不能参与计算、计算多快、以及占用多少显存。

#### 2.1 属性查看
- `.shape`：打印出报错位置前后所有张量的形状。
- `.device`：确认张量是在 cuda:0 还是 cpu（跨设备无法运算）。
- `.dtype`：确认是 float32 还是 bfloat16（不同精度有时也会导致特定的运算报错）。

In [7]:
x = torch.randn(2, 3, 1536, dtype=torch.bfloat16, device="cpu")

print(f"1. 形状 (Shape): {x.shape}")   # [Batch, Seq, Hidden]
print(f"2. 设备 (Device): {x.device}") # CPU 或 CUDA
print(f"3. 精度 (Dtype): {x.dtype}")   # float32, bfloat16 等

y = torch.zeros(3, 5, 1024, dtype=torch.float32, device='cuda')

print(f"1. 形状 (Shape): {y.shape}")   # [Batch, Seq, Hidden]
print(f"2. 设备 (Device): {y.device}") # CPU 或 CUDA
print(f"3. 精度 (Dtype): {y.dtype}")   # float32, bfloat16 等

1. 形状 (Shape): torch.Size([2, 3, 1536])
2. 设备 (Device): cpu
3. 精度 (Dtype): torch.bfloat16
1. 形状 (Shape): torch.Size([3, 5, 1024])
2. 设备 (Device): cuda:0
3. 精度 (Dtype): torch.float32


#### 2.2 形状变换 (Reshape / View)
在大模型中，数据经常需要在“三维”和“二维”之间切换（例如进入 Linear 层前需要压平）。

在处理大模型数据时，我们经常需要改变张量的“长相”以适应不同的计算工序。

##### 1. view：重塑形状 (Reshape)

`view` 的物理意义是在**不改变数据总量**的前提下，重新定义观察数据的“网格”。
- **输入**：目标形状的各个维度大小（Integers）。
- **输出**：形状改变后的新张量。
- **核心参数 `-1`**：
    - **含义**：这是一个“自动推导”占位符。当你确定了其它维度，但不确定剩余一个维度是多少时，填 `-1`，PyTorch 会根据元素总数自动算出该维度的值。
        

**工程案例：将 Batch 数据压平**

假设我们有 `[2, 4, 1536]` 的张量（2 句话，每句 4 个词，每个词 1536 维）。

- **需求**：将其送入线性层，线性层只接收二维输入 `[Token 总数, 维度]`。
- **操作**：`x.view(-1, 1536)`。
- **结果**：形状变为 `[8, 1536]`。这里的 `8` 就是由 $2 \times 4$ 自动推导出来的。
---

##### 2. unsqueeze：增加维度 (Dim Expansion)

`unsqueeze` 的物理意义是在指定位置“套上一层括号”，给张量升维。
- **输入 (dim)**：要在哪个位置插入维度。索引从 0 开始。
- **输出**：维度增加 1 后的新张量，新维度的大小恒等于 1。

**工程案例：单句变 Batch**
当你手动测试模型时，Tokenizer 可能会输出一个 `[Seq_Len]` 的向量（如长度为 4）。但模型要求输入必须有 Batch 维度，即 `[Batch, Seq_Len]`。
- **需求**：把 `[4]` 变成 `[1, 4]`。
- **操作**：`x.unsqueeze(0)`（在第 0 维插入一个位置）。
- **结果**：形状变为 `[1, 4]`

In [8]:
# 假设输入是 [2, 4, 1536] (Batch, Seq, Dim)
input_tensor = torch.randn(2, 4, 1536)

# 变换 A: 压平前两维，变为 [8, 1536]
# -1 表示自动计算该维度的大小
flattened = input_tensor.view(-1, 1536) 
print(f"压平后的形状: {flattened.shape}")

# 变换 B: 增加一个维度 (Unsqueeze)
# 模拟将单张图片变成一个 Batch
single_data = torch.randn(1536)
batch_ready = single_data.unsqueeze(0) 
print(f"增加 Batch 维度后: {batch_ready.shape}") # [1, 1536]

压平后的形状: torch.Size([8, 1536])
增加 Batch 维度后: torch.Size([1, 1536])


##### 总结表：常用维度变换对照

|**函数**|**核心参数**|**物理动作**|**形状变化示例**|
|---|---|---|---|
|**`view`**|`(d1, d2, ...)`|重新排列元素，总量不变|`[2, 8] -> [4, 4]`|
|**`view(-1, ...)`**|`(-1, d2)`|自动对齐剩余维度|`[2, 4, 512] -> [8, 512]`|
|**`unsqueeze`**|`dim=0`|在开头增加一个“1”维度|`[512] -> [1, 512]`|
|**`squeeze`**|`dim=0`|压缩掉大小为 1 的维度|`[1, 512] -> [512]`|


### 3.广播机制 (Broadcasting) —— 自动对齐的魔术

这是初学者最容易困惑的地方。为什么 `[2, 4, 1536]` 的张量可以和一个 `[1536]` 的张量相加？

- **规则**：如果两个张量的维度**从后往前看**，要么相等，要么其中一个是 1，PyTorch 就会自动通过“复制”来对齐形状。
- **工程意义**：在给 Embedding 加上位置编码，或者给所有 Token 加上同一个偏置项（Bias）时，广播机制能极大简化代码。

In [ ]:
# 模拟：给一句话里的每个词向量加上相同的偏移量
# 句子形状: [4, 1536] (4个词)
hidden_states = torch.randn(4, 1536)
# 偏移量形状: [1536]
bias = torch.ones(1536)

# 触发广播：bias 会被自动想象成 [4, 1536] 进行加法
output = hidden_states + bias 
print(f"相加后的形状: {output.shape}") # 依然是 [4, 1536]

**核心警示：Shape Mismatch 报错**

请务必记住这个报错逻辑： 如果你尝试让一个 `[2, 3]` 的矩阵和 `[4, 3]` 的矩阵做加法（且不符合广播规则），PyTorch 会报出 `RuntimeError: The size of tensor a (2) must match the size of tensor b (4)`。

**看到这个报错，第一反应应该是：打印 `tensor.shape`！**

## 3. Batch与并行化
我们来对比一下：用 for 循环一个一个算，和用 Batch 一次性算的效率差距有多大。


In [10]:
import time
# 模拟一个简单的矩阵乘法任务
matrix_a = torch.randn(1024, 1024, device="cuda")
matrix_b = torch.randn(1024, 1024, device="cuda")

# 场景 A：串行处理 32 次
start_time = time.time()
for _ in range(32):
    result = torch.matmul(matrix_a, matrix_b)
torch.cuda.synchronize() # 等待 GPU 计算完成
print(f"串行处理 32 次耗时: {time.time() - start_time:.4f} 秒")

# 场景 B：并行处理 (Batch Size = 32)
# 构造 [32, 1024, 1024] 的张量
batch_a = torch.randn(32, 1024, 1024, device="cuda")
batch_b = torch.randn(32, 1024, 1024, device="cuda")

start_time = time.time()
batch_result = torch.matmul(batch_a, batch_b)
torch.cuda.synchronize()
print(f"Batch 并行处理 32 份数据耗时: {time.time() - start_time:.4f} 秒")

串行处理 32 次耗时: 0.3274 秒
Batch 并行处理 32 份数据耗时: 0.0610 秒


## 4. 挑战练习：触发你的第一次 OOM！

不要害怕报错。现在我们要故意增加张量的大小，直到显存溢出（Out of Memory），学会识别这个深度学习中最常见的错误。

In [11]:
# 警告：运行此单元格会导致报错，这是正常的教学环节！
try:
    print("正在尝试分配一个巨大的张量以触发 OOM...")
    # 尝试分配一个 50GB 的张量（这通常会超过消费级显卡上限）
    oom_tensor = torch.randn(1024, 1024, 1024, 15, device="cuda")
except RuntimeError as e:
    print("\n🚨 捕获到错误！")
    print(f"错误信息: {e}")
    print("\n这就是 OOM 报错。当你看到 'OutOfMemoryError' 时，"
          "意味着你需要减小 Batch Size 或使用更低精度的模型。")

正在尝试分配一个巨大的张量以触发 OOM...

🚨 捕获到错误！
错误信息: CUDA out of memory. Tried to allocate 60.00 GiB. GPU 0 has a total capacity of 4.00 GiB of which 2.78 GiB is free. Of the allocated memory 404.23 MiB is allocated by PyTorch, and 21.77 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

这就是 OOM 报错。当你看到 'OutOfMemoryError' 时，意味着你需要减小 Batch Size 或使用更低精度的模型。
